##  Sentiment Analysis for Movie Dataset


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
from bs4 import BeautifulSoup
from os import path
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_curve, confusion_matrix,classification_report

from joblib import dump, load


from sklearn.externals.six import StringIO  
from IPython.display import Image  
from sklearn.tree import export_graphviz
import pydotplus
from subprocess import call

from sklearn.linear_model import LogisticRegression

[nltk_data] Downloading package stopwords to C:\Users\Leela
[nltk_data]     Bokka\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


ModuleNotFoundError: No module named 'sklearn.externals.six'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob

In [ ]:
trainPos = glob.glob('../data/aclImdb/train/pos/*.txt')
trainNeg = glob.glob('../data/aclImdb/train/neg/*.txt')
print("total reviews for training : {}".format(len(trainPos)+len(trainNeg)))
testPos = glob.glob('../data/aclImdb/test/pos/*.txt')
testNeg = glob.glob('../data/aclImdb/test/neg/*.txt')
print("total reviews for testing : {}".format(len(testPos)+len(testNeg)))

**Let's convert the text files into dataframe for further processing.**

In [ ]:
def extractText(folder):
    allText = []
    textFiles = []
    for fileName in folder:
        with open(fileName,'r') as txtfile:
            text = txtfile.read()
            text = text.lower()
            cleanText = BeautifulSoup(text,"html.parser").text
            allText.append(cleanText)
            textFiles.append(fileName)
    return allText, textFiles

In [ ]:
trainPosText,trainPosFiles = extractText(trainPos)
trainNegText,trainNegFiles = extractText(trainNeg)
testPosText,testPosFiles = extractText(testPos)
testNegText,testNegFiles = extractText(testNeg)

#for positive training files
dfTrainPos = pd.DataFrame(trainPosText,columns=['reviews'])
dfTrainPos['filename'] = trainPosFiles

#for negative training files
dfTrainNeg = pd.DataFrame(trainNegText,columns=['reviews'])
dfTrainNeg['filename'] = trainNegFiles

#for positive testing files
dfTestPos = pd.DataFrame(testPosText,columns=['reviews'])
dfTestPos['filename'] = testPosFiles

#for negative testing files
dfTestNeg = pd.DataFrame(testNegText,columns=['reviews'])
dfTestNeg['filename'] = testNegFiles


In [ ]:
dfTrainPos.head(5)

In [ ]:
dfTrainNeg.head(5)

In [ ]:
dfTestPos.head(5)

In [ ]:
dfTestNeg.head(5)

**let's build the wordcloud of the training data set initially for observing the frequency of words as well as their importance.**

In [ ]:
stpwords = list(STOPWORDS)
stpwords.extend(stopwords.words('english'))
stpwords.extend(['movie','movies','film','films',
                 'actor','actress','cinema',
                 'story','br','role','dramas','drama',
                 'characters','character','director'])
stpwords = set(stpwords)


In [ ]:
def wordCloud(text):
    wordcloud = WordCloud(stopwords=stpwords, background_color="white",width=800,height=800).generate(text)
    return wordcloud

In [ ]:
def plotWordCloud(wordcloud):
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.show()

In [ ]:
wordTrainPos = " ".join(review.strip() for review in dfTrainPos.reviews)
wordTrainNeg = " ".join(review.strip() for review in dfTrainNeg.reviews)

In [ ]:
wordcloudPos = wordCloud(wordTrainPos)
plotWordCloud(wordcloudPos)

In [ ]:
wordcloudNeg = wordCloud(wordTrainNeg)
plotWordCloud(wordcloudNeg)

In [ ]:
wordcloudPos.to_file('../reports/wordcloud/wordcloudpos.png')

In [ ]:
wordcloudNeg.to_file('../reports/wordcloud/wordcloudNeg.png')

**The wordcloud shown here shows the most common words that appear in the texts. However, the words shown here clearly donot define the sentiments of positive or negative reviews.**

**Let's see the weights of the words given by count vectorizer and visualize it.**

In [ ]:
stop_words = stopwords.words('english')
cntVectorizerPos = CountVectorizer(binary=False, stop_words=list(stpwords), min_df=5, max_df=0.90,max_features=40000)
cntVectorizerNeg = CountVectorizer(binary=False, stop_words=list(stpwords), min_df=5, max_df=0.90,max_features=40000)
dataTrainPos = cntVectorizerPos.fit_transform(dfTrainPos['reviews'])
dataTrainNeg = cntVectorizerNeg.fit_transform(dfTrainNeg['reviews'])

In [ ]:
featureNamesPos=cntVectorizerPos.get_feature_names()
featureNamesNeg=cntVectorizerNeg.get_feature_names()

In [ ]:
def getTopFeatures(features,n,featuresName):
    dictionary = {}
    featuresSum = features.sum(axis=0)[0,:]
    featuresSum = np.squeeze(np.asarray(featuresSum))
    sortedIndices = np.argsort(-1*featuresSum)[0:n]
    for index in sortedIndices:
        dictionary[featuresName[index]]=featuresSum[index]
    return dictionary

In [ ]:
dictionaryPos = getTopFeatures(dataTrainPos,500,featureNamesPos)
dictionaryNeg = getTopFeatures(dataTrainNeg,500,featureNamesNeg)
  

In [ ]:
def wordCloudFromFrequency(dictionary):
    return WordCloud(stopwords=stpwords, background_color="white",width=800,height=800).\
                                                generate_from_frequencies(dictionary)

In [ ]:
wordcloudPos = wordCloudFromFrequency(dictionaryPos)
plotWordCloud(wordcloudPos)


In [ ]:
wordcloudNeg = wordCloudFromFrequency(dictionaryNeg)
plotWordCloud(wordcloudNeg)

In [ ]:
wordcloudPos.to_file('../reports/wordcloud/wordcloudPoscount.png')
wordcloudNeg.to_file('../reports/wordcloud/wordcloudNegcount.png')

**As seen from the wordcloud above, we observe top 500 most repetitive words and they too may not represent the words that are important in classifying the sentiments.**

**let's use some of the classical approaches through which we can know the type of words that are used in classifying sentiments. let's update the label by our own. The positive sentiments are given label 1 and negative sentiments are given label 0.**

## Random Forest for Visualization

In [ ]:
dfTrainPos['label'] = 1
dfTrainNeg['label'] = 0
dfTestPos['label'] = 1
dfTestNeg['label'] = 0
trainDf = dfTrainPos.append(dfTrainNeg,ignore_index=True)
trainDf = trainDf.sample(frac=1)#shuffle the dataframe to randomize

testDf = dfTestPos.append(dfTestNeg,ignore_index=True)
testDf = testDf.sample(frac=1)
trainDf.head(5)

**We are considering the words that are present in at least 5 documents and rejecting the words that are present in 95% of the text files.**

In [ ]:
countVect = CountVectorizer(binary=False, stop_words=stop_words,
                             token_pattern="[a-zA-Z]{2,}",#only considering words which have 2 alphabets in minimum
                             min_df=5,max_df=0.95)
trainFeatures = countVect.fit_transform(trainDf.reviews)
features = countVect.get_feature_names()

In [ ]:
print("the number of words taken for training sentiments: {}".format(len(features)))


**lets choose the best hyperparameters.**

In [ ]:
params = {
    'max_depth':[5,6,7],
    'max_features':[0.9, 1.0],
    'n_estimators':[10,15]
}

rfClassifier = RandomForestClassifier(random_state=6,n_jobs=-1)
gridSearch = GridSearchCV(rfClassifier, params,verbose=3,n_jobs=-1)
gridSearch.fit(trainFeatures, trainDf.label)
print(gridSearch.best_estimator_)

**For the given range of parameters, the best hyperparameters are given and best estimator is obtained.**

In [ ]:
bestClassifier = gridSearch.best_estimator_
dump(bestClassifier, '../models/rfClassifier.joblib')

loadedModel = load('../models/rfClassifier.joblib')
rfTrainPredict = loadedModel.predict(trainFeatures)
score = accuracy_score(trainDf.label, rfTrainPredict)
print("the accuracy score for training set : {}".format(score))

# for testing data
testFeatures = countVect.transform(testDf.reviews)
rfTestPredict = loadedModel.predict(testFeatures)
score = accuracy_score(testDf.label, rfTestPredict)
print("the accuracy score for testing set : {}".format(score))
print(confusion_matrix(testDf.label,rfTestPredict))
print(classification_report(testDf.label,rfTestPredict))
precision, recall, threshold = precision_recall_curve(testDf.label,rfTestPredict)
plt.plot(recall, precision)
plt.xlabel('recall')
plt.ylabel('precision')
plt.show()

In [ ]:
featureImp = loadedModel.feature_importances_
indices = np.argsort(-featureImp)

trainWords = {}
for index in indices[0:100]:
    trainWords[features[index]] = featureImp[index]


In [ ]:
wordcloudAfterClassifier = wordCloudFromFrequency(trainWords)
plotWordCloud(wordcloudAfterClassifier)

In [ ]:
wordcloudAfterClassifier.to_file('../reports/wordcloud/wordcloudafterclassifier.png')

**The wordcloud above shows that after training, we have obtained the most important top 100 words that are used for classification. The words are mostly the adjective describing the positive and negative sentiments.**

In [ ]:
bestModel = loadedModel.estimators_[9]

In [ ]:
dot_data = StringIO()
export_graphviz(bestModel, out_file=dot_data, feature_names=features, 
               class_names=['negative','positive'], filled=True, rounded=True, 
                special_characters=True)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
Image(graph.create_png())

In [ ]:
graph.write_png('../reports/wordcloud/decisionTree.png')

**The decision tree above is one of the sections of random forest which shows the branching to positive and negative sentiments based on the occurrence of words.**

## Logistic regression for positive and negative words

In [ ]:
params = {
    'C':[0.07,0.09,0.1,0.2,0.3,0.5,1]
}

logisticModel = LogisticRegression(random_state=6,verbose=0,n_jobs=-1)
gridSearch = GridSearchCV(logisticModel, params,verbose=3,n_jobs=-1)
gridSearch.fit(trainFeatures, trainDf.label)
print(gridSearch.best_estimator_)

In [ ]:
posEg = {}
negEg = {}
bestModel = gridSearch.best_estimator_
indicesForPosEg = np.argsort(-1*bestModel.coef_)[:,0:100][0]
for i in indicesForPosEg[0:100]:
    posEg[features[i]] = bestModel.coef_[0,i]
indicesForNegEg = np.argsort(bestModel.coef_)[:,0:100][0]
for i  in indicesForNegEg[0:100]:
    negEg[features[i]] = -1*bestModel.coef_[0,i]

In [ ]:
bestModel = gridSearch.best_estimator_
dump(bestModel, '../models/logisticRegression.joblib')

loadedModel = load('../models/logisticRegression.joblib')

trainPredictLR = loadedModel.predict(trainFeatures)
testPredictLR = loadedModel.predict(testFeatures)

accuracyLRtrain = accuracy_score(trainDf.label, trainPredictLR)
accuracyLRtest = accuracy_score(testDf.label, testPredictLR)
print("the accuracy of training set is :{}".format(accuracyLRtrain))
print("the accuracy of testing set is :{}".format(accuracyLRtest))
print(confusion_matrix(testDf.label,testPredictLR))
print(classification_report(testDf.label,testPredictLR))
precision, recall, threshold = precision_recall_curve(testDf.label,testPredictLR)
plt.plot(recall, precision)
plt.xlabel('recall')
plt.ylabel('precision')
plt.show()

In [ ]:
wordcloudLogRegPos = wordCloudFromFrequency(posEg)
plotWordCloud(wordcloudLogRegPos)

In [ ]:
wordcloudLogRegPos.to_file('../reports/wordcloud/wordcloudLogRegressionPos.png')

In [ ]:
wordcloudLogRegNeg = wordCloudFromFrequency(negEg)
plotWordCloud(wordcloudLogRegNeg)

In [ ]:
wordcloudLogRegNeg.to_file('../reports/wordcloud/wordcloudLogRegressionNeg.png')

**The logistic regression showed the higher accurracy for classification and the wordclouds represent the important postive as well as negative words used for classification.**